# Simple Tokeniser — LLMs from Scratch

This notebook walks through building a simple text tokeniser step by step,
following Chapter 2 of *Build a Large Language Model (From Scratch)* by Sebastian Raschka.

Run each cell in order.

## Step 0: Install Dependencies

We use `uv` for fast dependency installation inside Colab.

In [ ]:
!pip install uv
!uv pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/requirements.txt --system

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 50.2 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 133 packages in 817ms
Prepared 8 packages in 1.37s
Uninstalled 1 package in 12ms
Installed 8 packages in 215ms
 + async-lru==2.3.0
 + jedi==0.20.0
 + json5==0.15.0
 + jupyter-builder==1.0.2
 + jupyter-lsp==2.3.1
 - jupyter-server==2.18.2
 + jupyter-server==2.20.0
 + jupyterlab==4.6.0
 + jupyterlab-server==2.28.0


## Step 1: Download the Dataset

We use *The Verdict*, a short story by Edith Wharton, as our sample corpus.
The file is downloaded only if it does not already exist.

In [ ]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
    print("Downloaded the-verdict.txt")
else:
    print("File already exists.")

Downloaded the-verdict.txt


In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Preview the first 200 characters
raw_text[:200]

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a'

In [ ]:
# Total number of characters in the text
len(raw_text)

20479

## Step 2: Basic Tokenisation with `re.split`

We explore how Python's `re.split` breaks a sentence into tokens.
The key idea is to **keep** delimiters (punctuation, whitespace) as separate tokens
by wrapping the pattern in a capture group `(...)`.

In [ ]:
import re

text = "Hello, world, This, is a test."

# Split on whitespace — keeps the whitespace token
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world,', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [ ]:
# Split on comma or period followed by a space
result = re.split(r'([,.]\s)', text)
print(result)

['Hello', ', ', 'world', ', ', 'This', ', ', 'is a test.']


In [ ]:
# Remove empty / whitespace-only tokens
result = [item for item in result if item.strip()]
print(result)

['Hello', ', ', 'world', ', ', 'This', ', ', 'is a test.']


## Step 3: Tokenise the Full Text

We now apply a richer regex pattern to the full story.
It handles commas, periods, colons, semicolons, question marks,
quotes, parentheses, double-dashes, and whitespace.

In [ ]:
result = re.split(r'([,.:;?"()\\]|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocessed = result

In [ ]:
# Total number of tokens
len(preprocessed)

4459

In [ ]:
# First 10 tokens
preprocessed[:10]

['I',
 'HAD',
 'always',
 'thought',
 'Jack',
 'Gisburn',
 'rather',
 'a',
 'cheap',
 'genius']

## Step 4: Build the Vocabulary

A vocabulary maps every unique token to a unique integer ID.
We sort the tokens alphabetically so the mapping is deterministic.

In [ ]:
all_words = sorted(set(preprocessed))
vocab = {word: i for i, word in enumerate(all_words)}
vocab_size = len(all_words)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1181


In [ ]:
# Look up the integer ID of a specific word
vocab["Jack"]

70

In [ ]:
# Build the reverse mapping: integer -> token
int_to_str = {i: s for s, i in vocab.items()}
int_to_str[57]

'Her'

## Step 5: `SimpleTokenizerV1` Class

We wrap the encode/decode logic inside a class:

- **`encode(text)`** — converts a string to a list of integer IDs
- **`decode(ids)`** — converts a list of integer IDs back to a string

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?"()\\]|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Remove spaces inserted before punctuation marks
        text = re.sub(r'\s+([,.:;?"()\\])', r'\1', text)
        return text

## Step 6: Test the Tokeniser

We create a tokeniser from our vocabulary and test it on a sample sentence
taken from the story itself.

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""

ids = tokenizer.encode(text)
print(ids)

[0, 69, 1029, 641, 566, 788, 11, 1175, 635, 11, 0, 83, 13, 43, 891, 1154, 796, 836, 13]


In [ ]:
# Decode the integer IDs back to text
tokenizer.decode(ids)

'" It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [ ]:
# Full roundtrip: encode then decode — should reproduce the original text
tokenizer.decode(tokenizer.encode(text))

'" It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

---
## Step 7: Using `tiktoken` — A Production-Grade Tokeniser

`tiktoken` is OpenAI's tokeniser, used by the GPT family of models.
Unlike our simple word-level approach, it uses **Byte Pair Encoding (BPE)**,
which learns common sub-word units from a large corpus.

Key differences from `SimpleTokenizerV1`:

| | `SimpleTokenizerV1` | `tiktoken` (GPT-2) |
|---|---|---|
| Method | Regex split on punctuation/whitespace | Byte Pair Encoding (BPE) |
| Vocab size | ~1,000 (only words seen in our text) | 50,257 |
| Unknown words | Raises `KeyError` | Always encodable (byte-level fallback) |
| Used by | This tutorial | GPT-2, GPT-3, GPT-4 |

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

# Load the GPT-2 encoding (same BPE vocab used by GPT-2 and GPT-3)
enc = tiktoken.get_encoding("gpt2")

print(f"tiktoken (GPT-2) vocabulary size: {enc.n_vocab}")
print(f"Our SimpleTokenizerV1 vocab size: {vocab_size}")

tiktoken (GPT-2) vocabulary size: 50257
Our SimpleTokenizerV1 vocab size: 1181


### Encode a string

We use the same sample sentence as before so you can compare outputs side by side.

In [ ]:
sample_text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""

token_ids = enc.encode(sample_text)
print(f"Token IDs  : {token_ids}")
print(f"Token count: {len(token_ids)}")

Token IDs  : [1, 1026, 338, 262, 938, 339, 13055, 11, 345, 760, 553, 198, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 9074, 13, 402, 271, 10899, 531, 351, 27322, 540, 11293, 13]
Token count: 33


### Inspect individual tokens

BPE splits words into sub-word pieces. Let's see exactly what each integer ID maps to.

In [ ]:
# Show each token ID alongside its decoded string
for tid in token_ids:
    print(f"{tid:>6}  →  {repr(enc.decode([tid]))}")

     1  →  '"'
  1026  →  'It'
   338  →  "'s"
   262  →  ' the'
   938  →  ' last'
   339  →  ' he'
 13055  →  ' painted'
    11  →  ','
   345  →  ' you'
   760  →  ' know'
   553  →  ',"'
   198  →  '\n'
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
   220  →  ' '
  9074  →  ' Mrs'
    13  →  '.'
   402  →  ' G'
   271  →  'is'
 10899  →  'burn'
   531  →  ' said'
   351  →  ' with'
 27322  →  ' pardon'
   540  →  'able'
 11293  →  ' pride'
    13  →  '.'


### Decode back to text

In [ ]:
# Decode the full list of IDs back to a string
enc.decode(token_ids)

'"It\'s the last he painted, you know,"\n           Mrs. Gisburn said with pardonable pride.'

### Full roundtrip — encode → decode

In [ ]:
# encode → decode should reproduce the original text exactly
enc.decode(enc.encode(sample_text))

'"It\'s the last he painted, you know,"\n           Mrs. Gisburn said with pardonable pride.'

### Bonus: Tokenise the entire story

Because tiktoken uses a 50k-word vocabulary with sub-word splitting,
it can handle any text — even words it has never seen — without raising an error.

In [ ]:
full_ids = enc.encode(raw_text)

print(f"Characters : {len(raw_text):,}")
print(f"Tokens     : {len(full_ids):,}")
print(f"First 10 IDs: {full_ids[:10]}")

Characters : 20,479
Tokens     : 5,145
First 10 IDs: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
